# 04. Rust End-to-End Parity Validation

This notebook captures the Rust/native backend evidence for the current worktree.

Read it after `start_here.ipynb` and `03_pystamps_verification.ipynb`. Those notebooks explain the general workflow; this one narrows the question to the Rust extension contract:

- prove the compiled Rust module is available and list the functions it exports;
- show the MATLAB StaMPS-backed audit artifact used as the parity oracle;
- rerun explicit `verify` checks against the recorded StaMPS golden roots;
- compare Python and Rust/native stage-2 outputs byte-for-byte or numerically;
- benchmark the stage-2 topofit kernel and show whether Rust is faster on this machine.

Speed claims in this notebook are measured locally. If the Rust extension is not built, the parity and speed cells report that explicitly instead of silently falling back.


In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import hashlib
import json
import os
import statistics
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from pystamps.kernels import describe_backend_matrix, run_stage2_topofit_kernel

REPO_ROOT = Path.cwd().resolve()
AUDIT_CONFIG = REPO_ROOT / 'notebooks' / '03_pystamps_verification.audit.yaml'
AUDIT_JSON = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_audit.json'
DATASET_STAGE8 = REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test_stage8diag'
DATASET_MAIN = REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test'
PROBE_ROOT = REPO_ROOT / 'tmp' / 'rust_parity_validation'

BASE_ENV = {
    **os.environ,
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'PYTHONPATH': str(REPO_ROOT),
}


def _ts(path: Path | None) -> str:
    if path is None or not path.exists():
        return '<missing>'
    return datetime.fromtimestamp(path.stat().st_mtime, UTC).isoformat()


def run_command(cmd: list[str], *, check: bool = True, env: dict[str, str] | None = None) -> subprocess.CompletedProcess[str]:
    t0 = time.perf_counter()
    completed = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=env or BASE_ENV,
        text=True,
        capture_output=True,
        check=False,
    )
    completed.elapsed_sec = time.perf_counter() - t0
    print('$', ' '.join(cmd))
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'command failed with exit code {completed.returncode}')
    return completed


native_import_error = None
native_spec = importlib.util.find_spec('pystamps.kernels._stage2_native')
try:
    native_mod = importlib.import_module('pystamps.kernels._stage2_native') if native_spec is not None else None
except Exception as exc:
    native_import_error = f'{type(exc).__name__}: {exc}'
    native_mod = None
native_path = Path(native_spec.origin) if native_spec is not None and native_spec.origin and native_mod is not None else None
src_path = REPO_ROOT / 'src' / 'lib.rs'

artifact_rows = [
    {'artifact': 'rust_source', 'path': str(src_path), 'modified_utc': _ts(src_path)},
    {'artifact': 'audit_config', 'path': str(AUDIT_CONFIG), 'modified_utc': _ts(AUDIT_CONFIG)},
    {'artifact': 'audit_json', 'path': str(AUDIT_JSON), 'modified_utc': _ts(AUDIT_JSON)},
]
if native_path is None:
    artifact_rows.append({'artifact': 'native_module', 'path': '<not importable>', 'modified_utc': '<missing>'})
else:
    artifact_rows.insert(0, {'artifact': 'native_module', 'path': str(native_path), 'modified_utc': _ts(native_path)})

pd.DataFrame(artifact_rows)


In [ ]:
backend_matrix = describe_backend_matrix()
stage2_topofit = backend_matrix.get('kernels', {}).get('stage2_topofit', {})
native_exports = sorted(name for name in dir(native_mod) if not name.startswith('_')) if native_mod is not None else []

pd.Series(
    {
        'python_executable': sys.executable,
        'python_version': sys.version.split()[0],
        'native_available': native_mod is not None,
        'native_origin': str(native_path) if native_path is not None else '<not importable>',
        'native_exports': ', '.join(native_exports) if native_exports else '<none>',
        'stage2_topofit_supported_backends': ', '.join(stage2_topofit.get('supported_backends', [])),
        'stage2_topofit_available_backends': ', '.join(stage2_topofit.get('available_backends', [])),
        'dataset_stage8diag_present': DATASET_STAGE8.exists(),
        'dataset_main_present': DATASET_MAIN.exists(),
        'audit_json_present': AUDIT_JSON.exists(),
        'native_import_error': native_import_error or '<none>',
        'native_build_hint': '<built>' if native_mod is not None else 'build/install the Rust extension, then rerun this notebook',
    }
)


## MATLAB StaMPS parity audit artifact

The expensive audit should be generated before opening this notebook. It uses `pystamps/data/audited_workflow_manifest.json` to select the maintained datasets and compares regenerated pySTAMPS outputs against the trusted MATLAB StaMPS golden roots recorded in the audit payload.

Use the manifest-backed command below. Avoid the older two-dataset command because the audited target set can grow.

```bash
OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 PYTHONPATH=.   uv run python scripts/validate_audit.py     --config notebooks/03_pystamps_verification.audit.yaml     --output inputs_and_outputs/validation_runs/latest_audit.json
```

A green audit means the native-backed run matched the active StaMPS oracle contract for every manifest target. The cells below re-display the artifact and rerun faster explicit verification against the `run_root` and `golden_root` pairs saved inside it.


In [ ]:
if AUDIT_JSON.exists():
    audit_payload = json.loads(AUDIT_JSON.read_text(encoding='utf-8'))
else:
    audit_payload = {
        'generated_at_utc': '<missing>',
        'completed': False,
        'interrupted': False,
        'ok': False,
        'failed_workflows': ['missing latest_audit.json'],
        'audits': [],
    }

audit_summary = pd.DataFrame(
    [
        {
            'dataset': audit['dataset'],
            'status': audit['status'],
            'ok': audit['ok'],
            'checked': audit['checked'],
            'run_root': audit['run_root'],
            'golden_root': audit['golden_root'],
            'run_source': audit['run_source'],
        }
        for audit in audit_payload['audits']
    ]
)
display(audit_summary if not audit_summary.empty else pd.DataFrame([{'audit': 'latest_audit.json missing or empty'}]))
pd.Series(
    {
        'generated_at_utc': audit_payload['generated_at_utc'],
        'completed': audit_payload['completed'],
        'interrupted': audit_payload['interrupted'],
        'ok': audit_payload['ok'],
        'failed_workflows': ', '.join(audit_payload['failed_workflows']) or '<none>',
    }
)


In [ ]:
verify_rows: list[dict[str, object]] = []
for audit in audit_payload['audits']:
    verify_cmd = [
        sys.executable,
        '-m',
        'pystamps.cli',
        '--config',
        str(AUDIT_CONFIG),
        'verify',
        '--run',
        str(audit['run_root']),
        '--golden',
        str(audit['golden_root']),
    ]
    completed = run_command(verify_cmd, check=False)
    verify_payload = json.loads(completed.stdout) if completed.stdout.strip().startswith('{') else {'ok': False, 'checked': 0, 'failed': []}
    verify_rows.append(
        {
            'dataset': audit['dataset'],
            'returncode': completed.returncode,
            'elapsed_sec': round(completed.elapsed_sec, 3),
            'ok': verify_payload['ok'],
            'checked': verify_payload['checked'],
            'failed_count': len(verify_payload['failed']),
        }
    )

verify_df = pd.DataFrame(verify_rows)
display(verify_df if not verify_df.empty else pd.DataFrame([{'verify': 'skipped because no audit rows were available'}]))


In [ ]:
pytest_expr = (
    'stage2_native_kernels_match_python_reference '
    'or stage4_stage7_stage8_native_kernels_match_python_reference '
    'or stage2_native_generic_matches_python_single_precision '
    'or stage2_native_generic_matches_python_single_precision_near_max_selector_regression '
    'or stage2_histogram_kernel_native_equal_spacing_matches_cpu'
)
if native_mod is None:
    pytest_completed = None
    pytest_result = pd.Series(
        {
            'returncode': '<skipped>',
            'selection': pytest_expr,
            'reason': 'native extension is not importable in this environment',
        }
    )
else:
    pytest_cmd = [
        sys.executable,
        '-m',
        'pytest',
        '-q',
        'tests/test_kernels_accelerated.py',
        '-k',
        pytest_expr,
    ]
    pytest_completed = run_command(pytest_cmd, check=False)
    pytest_result = pd.Series(
        {
            'returncode': pytest_completed.returncode,
            'elapsed_sec': round(pytest_completed.elapsed_sec, 3),
            'selection': pytest_expr,
        }
    )
pytest_result


In [ ]:
PROBE_ROOT.mkdir(parents=True, exist_ok=True)
python_probe = PROBE_ROOT / 'python_probe'
native_probe = PROBE_ROOT / 'native_probe'
force_probe = os.environ.get('PYSTAMPS_NOTEBOOK_FORCE_STAGE2_PROBE') == '1'

probe_base = [
    sys.executable,
    'scripts/stage2_patch1_probe.py',
    '--dataset',
    str(DATASET_STAGE8),
    '--patch',
    'PATCH_1',
]


def md5sum(path: Path) -> str:
    digest = hashlib.md5()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def ensure_probe(run_root: Path, backend: str) -> tuple[subprocess.CompletedProcess[str] | None, dict[str, str]]:
    pm1 = run_root / 'PATCH_1' / 'pm1.mat'
    should_run = force_probe or not pm1.exists()
    if should_run:
        completed = run_command(
            probe_base + ['--run-root', str(run_root), '--kernel-backend', backend],
            check=False,
        )
    else:
        completed = None
    payload = {
        'pm1_exists': str(pm1.exists()),
        'pm1_md5': md5sum(pm1) if pm1.exists() else '<missing>',
        'pm1_mtime_ns': str(pm1.stat().st_mtime_ns) if pm1.exists() else '<missing>',
        'elapsed_sec': f'{completed.elapsed_sec:.3f}' if completed is not None else '<reused>',
    }
    return completed, payload


python_completed, python_probe_payload = ensure_probe(python_probe, 'python')
if native_mod is None:
    native_completed = None
    native_probe_payload = {
        'pm1_exists': '<skipped>',
        'pm1_md5': '<native unavailable>',
        'pm1_mtime_ns': '<native unavailable>',
        'elapsed_sec': '<native unavailable>',
    }
else:
    native_completed, native_probe_payload = ensure_probe(native_probe, 'native')

probe_df = pd.DataFrame(
    [
        {
            'backend': 'python',
            'returncode': 0 if python_completed is None else python_completed.returncode,
            'reused_existing_run': python_completed is None,
            **python_probe_payload,
        },
        {
            'backend': 'native',
            'returncode': '<skipped>' if native_mod is None else (0 if native_completed is None else native_completed.returncode),
            'reused_existing_run': native_completed is None,
            **native_probe_payload,
        },
    ]
)
display(probe_df)


## Rust speed evidence

This benchmark compares the Python reference stage-2 topofit kernel with the compiled Rust/native implementation on the same synthetic, deterministic input. It is intentionally smaller than a full audit so it can be rerun interactively.

The parity columns prove the two backends return the same numerical results for the benchmark input. The speedup column is the local claim: values greater than `1.0` mean Rust is faster than the Python backend on this machine.


In [ ]:
bench_rows: list[dict[str, object]] = []
speed_summary = pd.Series(
    {
        'native_speedup_vs_python': '<not measured>',
        'rust_faster': False,
        'parity_max_abs': '<not measured>',
    }
)

if native_mod is None:
    bench_rows.append(
        {
            'backend': 'native',
            'ok': False,
            'reason': 'native extension is not importable; build it before measuring Rust speed',
        }
    )
else:
    rng = np.random.default_rng(42)
    n_ps = int(os.environ.get('PYSTAMPS_NOTEBOOK_BENCH_PS', '1024'))
    n_ifg = int(os.environ.get('PYSTAMPS_NOTEBOOK_BENCH_IFG', '64'))
    repeats = int(os.environ.get('PYSTAMPS_NOTEBOOK_BENCH_REPEATS', '5'))
    phase = rng.normal(size=(n_ps, n_ifg))
    cpxphase = np.exp(1j * phase).astype(np.complex64)
    bperp = rng.normal(loc=0.0, scale=150.0, size=(n_ps, n_ifg)).astype(np.float32)

    def bench_backend(backend: str) -> tuple[list[float], tuple[np.ndarray, ...]]:
        timings: list[float] = []
        output: tuple[np.ndarray, ...] | None = None
        for run_ix in range(repeats + 1):
            t0 = time.perf_counter()
            output = run_stage2_topofit_kernel(cpxphase, bperp, 1.5, backend=backend)
            elapsed = time.perf_counter() - t0
            if run_ix > 0:
                timings.append(elapsed)
        assert output is not None
        return timings, output

    python_runs, python_output = bench_backend('python')
    native_runs, native_output = bench_backend('native')
    parity_max_abs = max(
        float(np.nanmax(np.abs(np.asarray(py) - np.asarray(na))))
        for py, na in zip(python_output, native_output, strict=True)
    )
    python_mean = statistics.mean(python_runs)
    native_mean = statistics.mean(native_runs)
    speedup = python_mean / native_mean if native_mean > 0 else float('inf')
    bench_rows.extend(
        [
            {
                'backend': 'python',
                'ok': True,
                'mean_sec': python_mean,
                'min_sec': min(python_runs),
                'runs_sec': ', '.join(f'{value:.4f}' for value in python_runs),
                'speedup_vs_python': 1.0,
            },
            {
                'backend': 'native',
                'ok': True,
                'mean_sec': native_mean,
                'min_sec': min(native_runs),
                'runs_sec': ', '.join(f'{value:.4f}' for value in native_runs),
                'speedup_vs_python': speedup,
            },
        ]
    )
    speed_summary = pd.Series(
        {
            'native_speedup_vs_python': round(speedup, 3),
            'rust_faster': bool(speedup > 1.0),
            'parity_max_abs': parity_max_abs,
            'benchmark_shape': f'{n_ps} PS x {n_ifg} IFG',
            'repeats': repeats,
        }
    )

bench_df = pd.DataFrame(bench_rows)
display(bench_df)
speed_summary


In [ ]:
summary = pd.Series(
    {
        'audit_ok': bool(audit_payload['ok']),
        'verify_ok': bool(verify_df['ok'].all()) if not verify_df.empty and 'ok' in verify_df else False,
        'pytest_ok': False if pytest_completed is None else pytest_completed.returncode == 0,
        'stage2_probe_ok': (
            (python_completed is None or python_completed.returncode == 0)
            and (native_mod is not None)
            and (native_completed is None or native_completed.returncode == 0)
        ),
        'stage2_probe_md5_match': (
            python_probe_payload.get('pm1_exists') == 'True'
            and native_probe_payload.get('pm1_exists') == 'True'
            and python_probe_payload.get('pm1_md5') == native_probe_payload.get('pm1_md5')
        ),
        'native_speedup_vs_python': speed_summary.get('native_speedup_vs_python', '<not measured>'),
        'rust_speed_superior': speed_summary.get('rust_faster', False),
    }
)
display(summary)


## What to do with the result

- If the audit, verify, pytest, probe, and speed-parity checks are green, the native path is matching the active MATLAB StaMPS contract and the Rust kernel is faster for the measured stage-2 workload.
- If parity fails, use `03_pystamps_verification.ipynb` for generic mismatch interpretation before changing accelerated code.
- If speed is not superior, inspect backend availability, CPU thread settings, and benchmark shape before making performance claims.
